If the ligand signal is not recoverable in the data, we cannot expect scLEMBAS to learn perturbation separation. Rather, let's subset to the stronger perturbations in the dataset. 

Here, we rank order perturbations. Next, we select only those that are significantly separated.

1) Attempt 1: null distribution
2) Attempt 2: E-distance

In [1]:
import os

from tqdm import trange

import scanpy as sc
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist


import sys
sclembas_path = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas_path))
from scLEMBAS import io

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = '1'
os.environ["MKL_NUM_THREADS"] = '1'
os.environ["OPENBLAS_NUM_THREADS"] = '1'
os.environ["VECLIB_MAXIMUM_THREADS"] = '1'
os.environ["NUMEXPR_NUM_THREADS"] = '1'

In [3]:
tf_adata = io.read_tfad(os.path.join(data_path, 'interim', author + '_tf_activity_all.h5ad'))


/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
cat_col = 'cell_type'
pert_col = 'ligand'
ctrl_pert = 'CTRL'

latent_label = 'pca'


# E-distance attempt

In [5]:
import scperturb as scp
from joblib import parallel_backend


# import pertpy as pt

In [6]:
# filter PCA space to number of components (since scperturb edist just takes them all)
tf_adata_sub = tf_adata.copy()
n_components = tf_adata_sub.uns['pca']['pca_rank']
tf_adata_sub.obsm['X_pca'] = tf_adata_sub.obsm['X_pca'][:, :n_components]

In [ ]:
def bootstrap_median_ci(x, n_boot=5000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    boots = np.array([np.median(rng.choice(x, size=len(x), replace=True))
                      for _ in range(n_boot)])
    lo, hi = np.quantile(boots, [alpha/2, 1-alpha/2])
    return lo, hi

In [52]:
# get E-distance
dist = scp.edist(
    adata = tf_adata_sub, 
    obs_key = pert_col, 
    obsm_key = 'X_pca', 
    pwd = None, 
    dist = 'sqeuclidean', 
    sample_correct=True, 
    n_jobs = n_cores, 
    verbose = False
)

# aggregate by median across perturbations and rank-order accordingly
pert_strength = {}
for p in dist.index:
    row = dist.loc[p, :].drop(p)   # exclude self
    pert_strength[p] = row.median()
    
pert_strength = pd.DataFrame(data = {'perturbation': pert_strength.keys(), 'global_sep': pert_strength.values()})
pert_strength = pert_strength.sort_values(by = 'global_sep', ascending = False).reset_index(drop = True)

# get the 95% confidence interval of the median estimate for the control condition
ctrl_vec = dist.loc[ctrl_pert].drop(ctrl_pert).values
ctrl_lo, ctrl_hi = bootstrap_median_ci(ctrl_vec, n_boot=5000, alpha=0.05, seed=seed)

# filter for those perturbations that are greater than the upper bound of the confidence intervale
pert_strength['strong'] = pert_strength.global_sep > ctrl_hi

pert_strength

,perturbation,global_sep,strong
0,TGFB1,172.008469,True
1,IFNG,112.759664,True
2,IFNA2,80.925503,True
3,IL13,41.889279,True
4,BMP4,41.280189,True
5,IL17A,34.665664,True
6,OSM,34.049150,True
7,FGF10,29.411042,True
8,CHIR99021,27.376018,False
9,TNFA,20.737187,False


# centroids attempt

In [15]:
def compute_pairwise_pert_strength(
    adata,
    latent_label: str,
    cat_col: str,
    pert_col: str,
    n_components: int = None,
    min_cells: int = 20, # min cells in a condition for the perturbation to be considered in that cell type
    metric: str = "euclidean",
    randomize: bool = False, 
    seed: int = 888,
):
    """
    Rank order perturbation strength with the following procedure:
    
    Within each cell type:
    1) Aggregate to perturbation centroids by the mean
    2) Calculate the pairwise distance between perturbation centroids
    3) For each perturbation, take the median distance to all other perturbations
    
    Then:
    4) Aggregate each perturbation across cell types by the median
    
    The higher the final aggregated distance, the higher the perturbation strength. 
    """
    
    
    X_ls = adata.obsm["X_{}".format(latent_label)]
    md = adata.obs[[cat_col, pert_col]].copy().reset_index(drop=True)
    
    if n_components is None:
        if latent_label == 'pca':
            tf_adata.uns['pca']['pca_rank']
        elif latent_label == 'pls':
            tf_adata.uns['pls']['pls_mod'].n_components
            
    X_ls = X_ls[:, :n_components]

    X_ls = pd.DataFrame(X_ls)
    X_ls[cat_col] = md[cat_col].values
    X_ls[pert_col] = md[pert_col].values

    feature_cols = X_ls.columns[:-2]

    all_strength_records = []

    # ---- For each cell type, compute centroid distances ----
    for ct, df_ct in X_ls.groupby(cat_col, observed = False):
        if randomize:
            df_ct = df_ct.copy()
            np.random.seed(seed)
            df_ct[pert_col] = np.random.permutation(df_ct[pert_col].values)

        # centroids per perturbation within this cell type
        centroids = (
            df_ct.groupby(pert_col, observed = False)[feature_cols]
            .mean()
        )

        # ensure enough cells for reliability
        n_cells_per_ct = df_ct.groupby(pert_col, observed = False).size()
        valid_perts = n_cells_per_ct[n_cells_per_ct >= min_cells].index
        centroids = centroids.loc[valid_perts]

        if centroids.shape[0] < 2:
            continue

        pert_list = centroids.index.tolist()

        # pairwise distances
        dmat = cdist(centroids.values, centroids.values, metric=metric)
        df_d = pd.DataFrame(dmat, index=pert_list, columns=pert_list)

        # per-pert mean distance to all others
        for p in pert_list:
            row = df_d.loc[p, :].drop(p)   # exclude self
            all_strength_records.append({
                "cell_type": ct,
                "perturbation": p,
                "aggr_distance_to_others": row.median(),
                "n_perts_in_ct": len(pert_list)
            })


    strength_df = pd.DataFrame(all_strength_records)

    # ---- Aggregate across cell types ----
    pert_strength = (
        strength_df
        .groupby("perturbation", observed = False)["aggr_distance_to_others"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"aggr_distance_to_others": "global_sep"})
    )

    return pert_strength, strength_df

Drop the control perturbation from the assessment and rank order perturbations in PCA space. 

We drop the control because we are going to include it regardless.

In [16]:
# tf_adata_sub = tf_adata[tf_adata.obs[pert_col] != ctrl_pert].copy()
n_components = tf_adata.uns['pca']['pca_rank']

In [17]:
pert_strength, _ = compute_pairwise_pert_strength(
    adata = tf_adata_sub,
    latent_label = 'pca',
    cat_col = cat_col,
    pert_col = pert_col,
    n_components = n_components,
    min_cells = 20, # min cells in a condition for the perturbation to be considered in that cell type
    metric = "euclidean",
    randomize = False
)

# pert_strength.set_index('perturbation', inplace = True)
pert_strength

,perturbation,global_sep
0,IFNG,6.971737
1,IFNA2,6.345455
2,TGFB1,5.416403
3,IL13,4.294928
4,BMP4,4.289118
5,CHIR99021,3.764413
6,OSM,3.436168
7,IL17A,3.148278
8,TNFA,3.095596
9,LEP,2.895883


In [12]:
n_perm = 100

tf_adata_rand = tf_adata_sub.copy()
null_dist = []
for i in trange(n_perm):
    pert_strength_rand, _ = compute_pairwise_pert_strength(
        adata = tf_adata_rand,
        latent_label = 'pca',
        cat_col = cat_col,
        pert_col = pert_col,
        n_components = n_components,
        min_cells = 20, # min cells in a condition for the perturbation to be considered in that cell type
        metric = "euclidean",
        randomize = True, 
        seed = seed + i
    )
    null_dist.append(pert_strength_rand.set_index('perturbation').loc[pert_strength.index, 'global_sep'].values)
    
null_dist = pd.DataFrame(null_dist).T
null_dist.index = pert_strength.index.tolist()

# 
obs = pert_strength["global_sep"].to_numpy()[:, None]     
null = null_dist.to_numpy()                               
pvals = (1 + (null >= obs).sum(axis=1)) / (null.shape[1] + 1)

pert_strength['pval (>)'] = pvals

100%|█████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 38.90it/s]


Try E-distance instead..